# Week 16: Advanced MLOps & Distributed Deep Learning
## Notebook 06: Activation Checkpointing & Gradient Recomputation from Scratch

### 1. Conceptual Intuition & Real-World Use Case
When training deep neural networks with 50+ Transformer layers and long context sequences:
- Model weights take up a fixed amount of VRAM.
- However, **Intermediate Activations** (saved during the forward pass so they can be reused during backpropagation) scale with $O(L \cdot B \cdot S \cdot H)$ and rapidly cause GPU Out-Of-Memory (OOM) errors!

In 2016, Chen et al. introduced **Activation Checkpointing (Gradient Checkpointing)**:
- Instead of caching activations for every single layer, **discard intermediate activations** during the forward pass.
- Only retain a few periodic "checkpoints" (e.g. every $\sqrt{L}$ layers).
- During the backward pass, **recompute intermediate activations on-the-fly**!
- **Result:** Cuts activation memory from $O(L) \to O(\sqrt{L})$ at the cost of just ~25-30% extra compute!

---

### 2. Mathematical Foundations: Memory vs. Compute Trade-off
For an $L$-layer network:
1. **Standard Training:**
   - Forward Pass: Compute and store activations for all $L$ layers. Memory: $O(L)$.
   - Backward Pass: Free activations layer by layer.

2. **Checkpointing with Segments of Size $k$ ($k = \sqrt{L}$):**
   - Store activations at checkpoint boundaries only ($L / \sqrt{L} = \sqrt{L}$ checkpoints stored).
   - Recompute each block of $\sqrt{L}$ layers during backprop.
   - **Total Peak Memory:**
     $$\text{Peak Memory} = \sqrt{L} \text{ (checkpoints)} + \sqrt{L} \text{ (active recomputation)} = O(\sqrt{L})$$
   - Extra Compute: Exactly 1 extra forward pass per layer (+33% runtime overhead).

In [ ]:
def compute_activation_vram(layers, batch_size, seq_len, hidden_dim):
    # Activation bytes per layer: ~34 * B * S * H (standard Transformer layer)
    bytes_per_layer = 34 * batch_size * seq_len * hidden_dim
    total_standard_gb = (bytes_per_layer * layers) / (1024 ** 3)
    
    # Checkpointed Peak Memory: sqrt(L) checkpoints + sqrt(L) segment
    k = int(np.sqrt(layers))
    num_checkpoints = layers // k
    checkpointed_gb = (bytes_per_layer * (num_checkpoints + k)) / (1024 ** 3)
    
    savings = (total_standard_gb - checkpointed_gb) / total_standard_gb * 100
    return total_standard_gb, checkpointed_gb, savings

layers_list = [24, 48, 80]
B, S, H = 4, 4096, 4096  # Modern 70B parameter configuration

print(f"{'Layers':<8} {'Standard VRAM':<18} {'Checkpointed VRAM':<20} {'RAM Savings':<12}")
print("-" * 60)
for L in layers_list:
    std_gb, chk_gb, pct = compute_activation_vram(L, B, S, H)
    print(f"{L:<8} {std_gb:>10.1f} GB      {chk_gb:>12.1f} GB        {pct:>6.1f}%")

### 3. Step-by-Step Code Walkthrough
1. **Memory Explosion:** In an 80-layer model with 4K context, standard training requires over 170 GB of activation memory alone, crashing an 80GB A100.
2. **Checkpointed Compression:** Activation checkpointing slashes peak memory down to 38.3 GB (a 77.5% reduction), allowing training to proceed comfortably on a single GPU.

---

### 4. Industry Applications & Trade-offs
- **PyTorch Native:** Enabled via `torch.utils.checkpoint.checkpoint()`.
- **Trade-off:** Adds ~25% to 30% execution time due to recomputing forward passes during backpropagation.

---

### 5. Top Interview Questions & Answers
**Q1: What functions should NOT be enclosed in activation checkpoints?**
> **Answer:** Non-deterministic operations (like Dropout or Random Data Augmentation) must not be checkpointed naively, because recomputing them during backprop would sample different random masks! PyTorch solves this by stashing the RNG state of the checkpointed segment.

**Q2: How does Selective Activation Checkpointing differ from Full Checkpointing?**
> **Answer:** Full checkpointing discards all activations in a block. Selective Checkpointing (used in FlashAttention) only discards memory-heavy operations with low compute costs (e.g. Attention Softmax) while caching compute-heavy MLP matrix multiplications, achieving optimal speed/memory Pareto efficiency.